[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IshtVibhu/iv-Mesa-ABM-Tutorial/blob/main/NB_04_Matplotlib_Seaborn.ipynb)

# Python for Computational Economics
## Notebook 04 — Matplotlib & Seaborn: Economic Visualisation

**Prerequisites:** NB_01–NB_03

**What you will learn:**
- Line plots for time series (GDP, inflation, interest rates)
- Scatter plots and regression lines (Phillips curve, Okun's law)
- Histograms and density plots (wealth distributions)
- Heatmaps (input-output tables, correlation matrices)
- Visualising Sugarscape: wealth Lorenz curve and landscape grid

In [ ]:
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    import pandas as pd
except ImportError:
    !pip install matplotlib seaborn numpy pandas --quiet
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    import pandas as pd

# Use a clean, publication-ready style
plt.style.use("seaborn-v0_8-whitegrid")
print(f"Matplotlib {plt.matplotlib.__version__}, Seaborn {sns.__version__}")

---
## 1. Line Plots — Macro Time Series

In [ ]:
np.random.seed(0)
years = np.arange(2000, 2024)
gdp_growth  = np.array([4.1, 1.0, 1.7, 2.9, 3.8, 3.5, 2.9, 1.9, -0.1, -2.5,
                         2.5, 1.6, 2.2, 1.8, 2.5, 2.9, 1.6, 2.3, 2.9, 2.3,
                        -3.4, 5.9, 2.1, 2.5])
fed_rate    = np.array([6.5, 1.75,1.75,1.0, 2.25,4.25,5.25,5.25,2.0, 0.25,
                         0.25,0.25,0.25,0.25,0.25,0.5, 1.5, 2.5, 2.5, 1.75,
                         0.25,0.25,4.25,5.25])

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# GDP growth panel
axes[0].bar(years, gdp_growth,
            color=["#d62728" if g < 0 else "#1f77b4" for g in gdp_growth],
            width=0.7)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_ylabel("GDP Growth (%)")
axes[0].set_title("US GDP Growth and Federal Funds Rate (2000–2023)")

# Fed rate panel
axes[1].plot(years, fed_rate, color="#ff7f0e", linewidth=2, marker="o", markersize=4)
axes[1].set_ylabel("Fed Funds Rate (%)")
axes[1].set_xlabel("Year")

# Shade recession years
for ax in axes:
    for y, g in zip(years, gdp_growth):
        if g < 0:
            ax.axvspan(y - 0.5, y + 0.5, alpha=0.15, color="red")

plt.tight_layout()
plt.savefig("macro_timeseries.png", dpi=100, bbox_inches="tight")
plt.show()
print("Figure saved as macro_timeseries.png")

---
## 2. Scatter Plots — Phillips Curve

In [ ]:
np.random.seed(7)
# Simulated Phillips curve data: negative relationship with noise
unemployment = np.linspace(3, 10, 30) + np.random.normal(0, 0.3, 30)
inflation    = 8 - 0.8 * unemployment + np.random.normal(0, 0.5, 30)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(unemployment, inflation, alpha=0.7, s=60, color="steelblue", label="Observations")

# Fit a regression line
coeffs = np.polyfit(unemployment, inflation, deg=1)
u_line = np.linspace(unemployment.min(), unemployment.max(), 100)
ax.plot(u_line, np.polyval(coeffs, u_line), color="#d62728", linewidth=2,
        label=f"OLS fit: π = {coeffs[0]:.2f}·u + {coeffs[1]:.2f}")

ax.set_xlabel("Unemployment Rate (%)")
ax.set_ylabel("Inflation Rate (%)")
ax.set_title("Phillips Curve (simulated data)")
ax.legend()
plt.tight_layout()
plt.savefig("phillips_curve.png", dpi=100, bbox_inches="tight")
plt.show()

---
## 3. Wealth Distribution — Histogram and Lorenz Curve

In [ ]:
np.random.seed(42)

# Simulate a right-skewed wealth distribution (log-normal is typical)
wealth = np.random.lognormal(mean=3.0, sigma=1.2, size=1000)

def lorenz_curve(w):
    w_sorted = np.sort(w)
    cumulative_wealth = np.cumsum(w_sorted) / w_sorted.sum()
    cumulative_pop    = np.linspace(0, 1, len(w_sorted))
    return cumulative_pop, cumulative_wealth

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Histogram
ax1.hist(wealth, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
ax1.axvline(np.mean(wealth), color="red", linestyle="--", label=f"Mean = {np.mean(wealth):.0f}")
ax1.axvline(np.median(wealth), color="orange", linestyle="--", label=f"Median = {np.median(wealth):.0f}")
ax1.set_xlabel("Wealth")
ax1.set_ylabel("Number of Agents")
ax1.set_title("Wealth Distribution (log-normal)")
ax1.legend()

# Lorenz curve
pop, cum_wealth = lorenz_curve(wealth)
ax2.plot(pop, cum_wealth, color="steelblue", linewidth=2, label="Lorenz curve")
ax2.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect equality")
ax2.fill_between(pop, pop, cum_wealth, alpha=0.2, color="red", label="Area of inequality")

# Compute Gini from Lorenz
gini = 1 - 2 * np.trapz(cum_wealth, pop)
ax2.set_xlabel("Cumulative Population Share")
ax2.set_ylabel("Cumulative Wealth Share")
ax2.set_title(f"Lorenz Curve  (Gini = {gini:.3f})")
ax2.legend()

plt.tight_layout()
plt.savefig("wealth_distribution.png", dpi=100, bbox_inches="tight")
plt.show()

---
## 4. Heatmap — Correlation Matrix of Macro Variables

In [ ]:
np.random.seed(5)
n = 50

# Construct correlated macro variables
base       = np.random.normal(0, 1, n)
gdp_growth = base + np.random.normal(0, 0.3, n)
employment = 0.6 * base + np.random.normal(0, 0.5, n)
investment = 0.8 * base + np.random.normal(0, 0.4, n)
inflation  = -0.3 * base + np.random.normal(0, 0.6, n) + 2
interest   = 0.5 * inflation + np.random.normal(0, 0.3, n)

macro_df = pd.DataFrame({
    "GDP Growth":  gdp_growth,
    "Employment":  employment,
    "Investment":  investment,
    "Inflation":   inflation,
    "Interest Rate": interest,
})

corr = macro_df.corr().round(2)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1, vmax=1,
    square=True,
    ax=ax
)
ax.set_title("Correlation Matrix of Macro Variables")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=100, bbox_inches="tight")
plt.show()

---
## 5. Sugarscape Landscape Visualisation

In [ ]:
def make_landscape(size=50, peaks=None, peak_height=4, radius=15):
    if peaks is None:
        peaks = [(15, 15), (35, 35)]
    grid = np.zeros((size, size))
    for r in range(size):
        for c in range(size):
            for pr, pc in peaks:
                d = np.sqrt((r - pr)**2 + (c - pc)**2)
                grid[r, c] = max(grid[r, c], peak_height - d * peak_height / radius)
    return np.clip(grid, 0, peak_height).round().astype(int)

sugar = make_landscape(50, peaks=[(15, 15), (35, 35)])
spice = make_landscape(50, peaks=[(15, 35), (35, 15)])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, grid, title, cmap in zip(
    axes,
    [sugar, spice],
    ["Sugar Landscape", "Spice Landscape"],
    ["YlOrBr", "Greens"]
):
    im = ax.imshow(grid, cmap=cmap, interpolation="nearest", vmin=0, vmax=4)
    plt.colorbar(im, ax=ax, label="Capacity (0–4)")
    ax.set_title(title)
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")

plt.suptitle("Sugarscape G1MT Resource Landscapes", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig("sugarscape_landscape.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Your Turn — Gini Over Time Plot

Using the Sugarscape simulation from NB_03 (re-run below), create a line chart showing the **Gini coefficient of wealth** at each time step. Add a horizontal dashed line at the initial Gini value and label it "Initial inequality".

In [ ]:
# Re-create simulation data
np.random.seed(1)
N_AGENTS = 200
N_STEPS  = 30
wealth   = np.random.lognormal(3.0, 0.5, N_AGENTS)

def gini(w):
    w_sorted = np.sort(w)
    n = len(w_sorted)
    return (2 * np.sum(np.arange(1, n+1) * w_sorted) / (n * w_sorted.sum())) - (n+1)/n

gini_series = [gini(wealth)]
for _ in range(N_STEPS):
    wealth *= np.random.lognormal(0, 0.15, N_AGENTS)
    wealth  = np.clip(wealth, 0.01, None)
    gini_series.append(gini(wealth))

print(f"Gini at step 0:  {gini_series[0]:.3f}")
print(f"Gini at step 30: {gini_series[-1]:.3f}")

In [ ]:
# Your Gini-over-time plot here


In [ ]:
#@title Solution
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(N_STEPS + 1), gini_series, color="steelblue", linewidth=2, marker="o", markersize=4)
ax.axhline(gini_series[0], color="gray", linestyle="--", label="Initial inequality")
ax.set_xlabel("Time Step")
ax.set_ylabel("Gini Coefficient")
ax.set_title("Wealth Inequality Over Time (Sugarscape simulation)")
ax.legend()
plt.tight_layout()
plt.show()

---
## Summary

| Chart type | When to use | Key function |
|---|---|---|
| Line / bar | Time series, trends | `ax.plot()`, `ax.bar()` |
| Scatter | Two-variable relationships | `ax.scatter()` |
| Histogram | Distribution of a single variable | `ax.hist()` |
| Lorenz curve | Inequality measurement | `ax.plot()` with cumulative sums |
| Heatmap | Correlation matrices, grids | `sns.heatmap()` |
| imshow | 2-D arrays (landscapes) | `ax.imshow()` |

**Next:** NB_05_SciPy.ipynb — optimisation, integration, and ODEs for economic models.